# Relatório operacional de pré-decolagem

Notebook didático da issue #4. Ele executa cenários determinísticos do repositório e integra leitura, validação, energia e decisão. As faixas usadas são hipóteses didáticas; não representam parâmetros certificados de uma nave real.

## Preparação

O notebook funciona quando aberto na raiz do repositório ou dentro da pasta `notebooks/`. Não requer API key nem acesso à internet.

In [1]:
from pathlib import Path
import sys

CANDIDATOS = (Path.cwd(), Path.cwd().parent)
RAIZ = next(p for p in CANDIDATOS if (p / 'src').is_dir())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.apresentacao import formatar_resultado
from src.missao import LIMITES_PADRAO, executar_cenario

print('Repositório: raiz do projeto')
print(f'Limites didáticos: {LIMITES_PADRAO}')

Repositório: raiz do projeto
Limites didáticos: {'temperatura_interna_c': (15, 30), 'temperatura_externa_c': (-150, 120), 'energia_pct': (50, 100), 'pressao_tanque_kpa': (90, 110)}


## Apresentação dos resultados

A orquestração devolve dados estruturados e `formatar_resultado` os converte em texto legível, sem recalcular energia ou alterar a decisão.

## Cenário nominal

O cenário nominal deve liberar a decolagem.

In [2]:
nominal = executar_cenario(RAIZ / 'dados' / 'nominal.json', LIMITES_PADRAO)
print(formatar_resultado(nominal))

Cenário: nominal.json
Decisão: PRONTO PARA DECOLAR
Nenhuma falha operacional identificada.
Energia inicial: 80.00 kWh
Perdas: 4.00 kWh
Energia útil: 76.00 kWh
Saldo após decolagem: 56.00 kWh
Autonomia estimada: 5.60 h


## Falha de temperatura

A leitura interna acima da faixa segura é válida como dado, mas a decisão deve abortar e informar o motivo.

In [3]:
falha_temperatura = executar_cenario(RAIZ / 'dados' / 'falha_temperatura.json', LIMITES_PADRAO)
print(formatar_resultado(falha_temperatura))

Cenário: falha_temperatura.json
Decisão: DECOLAGEM ABORTADA
Motivos:
- temperatura_interna_c 31 acima do maximo de 30
Energia inicial: 80.00 kWh
Perdas: 4.00 kWh
Energia útil: 76.00 kWh
Saldo após decolagem: 56.00 kWh
Autonomia estimada: 5.60 h


## Falha energética

A telemetria é válida, porém o saldo depois de perdas e consumo é insuficiente.

In [4]:
falha_energia = executar_cenario(RAIZ / 'dados' / 'falha_energia.json', LIMITES_PADRAO)
print(formatar_resultado(falha_energia))

Cenário: falha_energia.json
Decisão: DECOLAGEM ABORTADA
Motivos:
- Energia insuficiente: saldo de -4.0 kWh
Energia inicial: 80.00 kWh
Perdas: 4.00 kWh
Energia útil: 76.00 kWh
Saldo após decolagem: -4.00 kWh
Autonomia temporal não calculada.


## Integração concluída

O notebook usa os módulos de missão, validação, energia, verificação e apresentação. A geração por IA é opcional: os JSONs fixos mantêm esta demonstração reproduzível sem credenciais nem internet.